In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from delta.tables import DeltaTable

dbutils.widgets.text("incremental_flag","0")
incremental_flag = dbutils.widgets.get("incremental_flag")


In [0]:
df_src = spark.read.table("sales.silver.customers")
df_src.schema

In [0]:
if spark.catalog.tableExists("sales.gold.customers"):
    df_sink = spark.sql("select * from sales.gold.customers")
else:
    df_sink = spark.sql("select *,0 dim_customer_key,0 create_date, 0 update_date from sales.silver.customers where 1 = 0")
df_sink.display()

In [0]:
df_sink = df_sink.withColumnRenamed("dim_customer_key","old_dim_customer_key") \
                .withColumnRenamed("customer_id","old_customer_id") \
                .withColumnRenamed("create_date","old_create_date") \
                .withColumnRenamed("update_date","old_update_date")
df_sink.display()


In [0]:
df_join = df_src.join(df_sink,df_src.customer_id == df_sink.old_customer_id,'left').select(df_src["*"],df_sink.old_dim_customer_key,df_sink.old_create_date,df_sink.old_update_date)
df_join.display()

In [0]:
df_old = df_join.where(df_join.old_dim_customer_key.isNotNull())
df_new = df_join.where(df_join.old_dim_customer_key.isNull())
display(df_old)
display(df_new)

Prepraing old_df


In [0]:
df_old = df_old.drop("old_customer_id","old_update_date")
df_old = df_old.withColumnRenamed("old_dim_customer_key","dim_customer_key")
df_old = df_old.withColumnRenamed("old_create_date","create_date")
df_old = df_old.withColumn("create_date",F.to_timestamp(df_old['create_date']))
df_old = df_old.withColumn("update_date",F.current_timestamp())
df_old.display()





Preparing df_new

In [0]:
df_new  = df_new.drop("old_customer_id","old_dim_customer_key","old_create_date","old_update_date")
df_new = df_new.withColumn("create_date",F.current_timestamp())
df_new = df_new.withColumn("update_date",F.current_timestamp())
df_new.display()


In [0]:
df_new = df_new.withColumn("dim_customer_key",F.monotonically_increasing_id() + F.lit(1))
df_new.display()


In [0]:
if (incremental_flag == "0"):
    max_value = 0
else:
    if spark.catalog.tableExists("sales.gold.cutomers"):
        df = spark.sql("select max(dim_customer_key) from sales.gold.customers")
        max_value = df.collect()[0][0]
print(max_value)


In [0]:
df_new = df_new.withColumn("dim_customer_key",F.lit(max_value) + df_new["dim_customer_key"])
df_new.display()


In [0]:
df_final = df_new.unionByName(df_old)
df_final.display()


In [0]:
if spark.catalog.tableExists("sales.gold.customers"):
    delta_table = DeltaTable.forName(spark,"sales.gold.customers")
    delta_table.alias("t").merge(df_final.alias("s"),"t.dim_customer_key = s.dim_customer_key").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
else:
    df_final.write.format("delta").mode("append").option("path","abfss://sales@adls7786.dfs.core.windows.net/gold/customers").saveAsTable("sales.gold.customers")

In [0]:
%sql

select * from sales.gold.customers;